<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-02-grafos/16%20-%20Problemas%20Eulerianos%20e%20Inspecao%20de%20Infraestrutura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 16 - Notebook: Circuitos Eulerianos e Inspeção de Manutenção na Manufatura

Neste notebook modelamos a malha física das passarelas de manutenção (grafo não-dirigido) e implementamos o **Algoritmo de Hierholzer** para gerar a rota ótima de inspeção do técnico ou robô industrial autônomo, garantindo que todas as esteiras sejam inspecionadas exatamente uma vez.


In [1]:
from collections import defaultdict
from typing import List, Tuple, Dict, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionários em tabela ASCII alinhada."""
    if not dados:
        return "Tabela Vazia"
    cols = list(dados[0].keys())
    w = {c: max(len(c), max(len(str(r.get(c, ''))) for r in dados)) for c in cols}
    head = " | ".join(f"{c:<{w[c]}}" for c in cols)
    div = "-+-".join("-" * w[c] for c in cols)
    rows = [" | ".join(f"{str(r.get(c, '')):<{w[c]}}" for c in cols) for r in dados]
    return "\n".join([head, div] + rows)

class GrafoInspecaoEulerianoFMS:
    """
    Modelagem da malha não-dirigida de passarelas de manutenção e esteiras da FMS.
    Implementa verificação de graus pares e o Algoritmo de Hierholzer.
    """
    def __init__(self):
        self.adj = defaultdict(list)
        self.passarelas = []

    def adicionar_passarela(self, u: str, v: str, id_p: str, comp_m: float = 2.0):
        self.adj[u].append((v, id_p))
        self.adj[v].append((u, id_p))
        self.passarelas.append({"ID": id_p, "Posto_A": u, "Posto_B": v, "Comprimento_m": comp_m})

    def verificar_euleriano(self) -> Tuple[bool, List[str], List[Dict[str, Any]]]:
        relatorio_graus = []
        impares = []
        for v in sorted(self.adj.keys()):
            deg = len(self.adj[v])
            if deg % 2 != 0:
                impares.append(v)
            relatorio_graus.append({"Posto": v, "Grau": deg, "Paridade": "PAR" if deg % 2 == 0 else "ÍMPAR"})
        return len(impares) == 0, impares, relatorio_graus

    def calcular_circuito_hierholzer(self, inicio: str) -> List[str]:
        adj_copia = {u: list(viz) for u, viz in self.adj.items()}
        pilha = [inicio]
        circuito = []
        while pilha:
            u = pilha[-1]
            if adj_copia[u]:
                v, id_e = adj_copia[u].pop()
                adj_copia[v].remove((u, id_e))
                pilha.append(v)
            else:
                circuito.append(pilha.pop())
        circuito.reverse()
        return circuito

# Instanciação da Malha de Manutenção Preventiva da FMS
g_insp = GrafoInspecaoEulerianoFMS()

# 1. Anel Periférico de Passarelas (Conectando a base de ferramentas a toda a linha)
g_insp.adicionar_passarela("BASE-MANUT", "SILO-100",  "p1",  1.5)
g_insp.adicionar_passarela("SILO-100",   "INSP-200",  "p2",  1.2)
g_insp.adicionar_passarela("INSP-200",   "DIV-201",   "p3",  1.0)
g_insp.adicionar_passarela("DIV-201",    "CX-301",    "p4",  0.8)
g_insp.adicionar_passarela("CX-301",     "CX-302",    "p5",  1.0)
g_insp.adicionar_passarela("CX-302",     "CX-303",    "p6",  1.0)
g_insp.adicionar_passarela("CX-303",     "DESC-304",  "p7",  1.0)
g_insp.adicionar_passarela("DESC-304",   "BASE-MANUT","p8",  2.5)

# 2. Malhas Transversais Intermediárias Balanceadas (Garantindo deg(v) = 4 nos nós críticos)
g_insp.adicionar_passarela("SILO-100",   "DIV-201",   "p9",  1.8)
g_insp.adicionar_passarela("DIV-201",    "DESC-304",  "p10", 2.2)
g_insp.adicionar_passarela("DESC-304",   "INSP-200",  "p11", 2.0)
g_insp.adicionar_passarela("INSP-200",   "SILO-100",  "p12", 1.2)

# Auditoria de paridade dos vértices
eh_euleriano, postos_impares, tabela_graus = g_insp.verificar_euleriano()

print("=== 1. AUDITORIA TOPOLÓGICA DE GRAUS (TEOREMA DE EULER) ===")
print(formatar_tabela(tabela_graus))
print(f"\nGrafo é Euleriano: {eh_euleriano} | Postos com Grau Ímpar: {postos_impares}")

# Cálculo da rota de inspeção autônoma
circuito_inspecao = g_insp.calcular_circuito_hierholzer("BASE-MANUT")
print("\n=== 2. CIRCUITO EULERIANO DE MANUTENÇÃO PREVENTIVA (HIERHOLZER) ===")
print(" -> ".join(circuito_inspecao))

# Asserções determinísticas
assert eh_euleriano is True, "Erro: Malha possui estações de grau ímpar!"
assert len(postos_impares) == 0
assert circuito_inspecao[0] == "BASE-MANUT" and circuito_inspecao[-1] == "BASE-MANUT"
assert len(circuito_inspecao) == len(g_insp.passarelas) + 1, "Erro: Nem todas as passarelas foram percorridas!"
print("\n[OK] Circuito euleriano de inspeção da FMS validado com sucesso!")

=== 1. AUDITORIA TOPOLÓGICA DE GRAUS (TEOREMA DE EULER) ===
Posto      | Grau | Paridade
-----------+------+---------
BASE-MANUT | 2    | PAR     
CX-301     | 2    | PAR     
CX-302     | 2    | PAR     
CX-303     | 2    | PAR     
DESC-304   | 4    | PAR     
DIV-201    | 4    | PAR     
INSP-200   | 4    | PAR     
SILO-100   | 4    | PAR     

Grafo é Euleriano: True | Postos com Grau Ímpar: []

=== 2. CIRCUITO EULERIANO DE MANUTENÇÃO PREVENTIVA (HIERHOLZER) ===
BASE-MANUT -> DESC-304 -> INSP-200 -> SILO-100 -> DIV-201 -> DESC-304 -> CX-303 -> CX-302 -> CX-301 -> DIV-201 -> INSP-200 -> SILO-100 -> BASE-MANUT

[OK] Circuito euleriano de inspeção da FMS validado com sucesso!
